# Cut Stability Experiment

One-time read-only companion to loop_detection.ipynb: nothing here removes
comparisons. Defaults follow the earlier rounds' findings: public stored
ratings (win probability against an average image, which damps
barely-compared images) combined with the gap-first candidate rule, oldest
vote as tiebreak. Every pass starts from those defaults and each cell changes
exactly one of them; the final section
replays the stored history in 50 seeded random orders and counts how often
each pair keeps getting cut. Only the last cell carries statistics; the
others show images.

In [ ]:
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)

configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)

RUN_COUNT: int = 50

SHOW_TOP: int = 10

In [ ]:
import time

from comfyui_image_scorer.adapters.cli.deps import CLIDeps, build_cli_deps
from comfyui_image_scorer.application.services.graph_service import CrystalGraph
from comfyui_image_scorer.domain.graph.chain_manager import ChainManager
from comfyui_image_scorer.domain.graph.link_proxy import LinkProxy
from comfyui_image_scorer.domain.graph.node_proxy import NodeProxy

t0: float = time.perf_counter()
deps: CLIDeps = build_cli_deps()
graph: CrystalGraph = deps.graph
graph.rebuild_from_database()

nodes: list[NodeProxy] = graph.get_all_nodes()
links: list[LinkProxy] = graph.get_all_links()
# experiment-only access to the chain manager the graph already owns
cm: ChainManager = graph._chain

logger.info(
    "Loaded %d comparisons across %d images",
    len(links),
    len(nodes),
    start_timer=t0,
)

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    CutGraph,
    build_cut_graph,
)

cut_graph: CutGraph = build_cut_graph(cm, links)
sizes: list[int] = [len(members) for _, members in cut_graph.cyclic_sccs]
logger.info(
    "Tarjan: %d cyclic SCCs covering %d nodes, top sizes %s",
    len(sizes),
    len(cut_graph.cyclic_members),
    sizes[:20],
)

## Score forms (rule: gap first, oldest tiebreak)

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "gap_ts", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default: public stored ratings, gap first, oldest tiebreak")

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "trueskill")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "gap_ts", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default plus trueskill instead of public")

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "position")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "gap_ts", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default plus position score instead of public")

## Rules (public stored ratings)

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "ts_gap", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default plus oldest-first rule")

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "gap_random", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default plus random tiebreak on gap")

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "sigma", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default plus sigma rule")

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "random", prune=False, lower_bound=False, seed=SEED
)
show_cut_grid(result.entries, SHOW_TOP, "default plus uniform random rule")

## Diagnostics (defaults plus one diagnostic)

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "gap_ts", prune=True, lower_bound=False, seed=SEED
)
logger.info(
    "Pruned %d redundant cut(s) of %d picked",
    result.pruned_count,
    len(result.entries),
)
show_cut_grid(result.entries, SHOW_TOP, "default plus redundancy prune")

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    proxy_scores,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import SEED, show_cut_grid

scores, sigmas = proxy_scores(nodes, "public")
result: PassResult = run_cut_pass(
    cut_graph, scores, sigmas, "gap_ts", prune=False, lower_bound=True, seed=SEED
)
for stat in result.scc_stats:
    if stat.lower_bound is None:
        continue
    optimal: str = " (provably optimal)" if stat.lower_bound == stat.cuts else ""
    logger.info(
        "SCC %d (n=%d): %d cut(s), lower bound %d%s",
        stat.scc_id,
        stat.size,
        stat.cuts,
        stat.lower_bound,
        optimal,
    )
show_cut_grid(result.entries, SHOW_TOP, "default plus lower bound")

## 50 random replays with the defaults (public stored ratings, gap first, oldest tiebreak)

In [ ]:
import time

from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    run_cut_pass,
)
from comfyui_image_scorer.experiments.cut_stability_lib import (
    SEED,
    PairStats,
    aggregate_runs,
    consensus_pairs,
    replay_scores,
    report_stats,
    show_frequency_histogram,
    show_pair_grid,
    verify_acyclic,
)

t0 = time.perf_counter()
runs: list[PassResult] = []
for run_index in range(RUN_COUNT):
    scores, sigmas = replay_scores(links, SEED + run_index)
    runs.append(
        run_cut_pass(
            cut_graph,
            scores,
            sigmas,
            "gap_ts",
            prune=False,
            lower_bound=False,
            seed=SEED + run_index,
        )
    )
    if (run_index + 1) % 10 == 0:
        logger.info("Replay %d/%d done", run_index + 1, RUN_COUNT)

stats: list[PairStats] = aggregate_runs(runs, RUN_COUNT)
cut_counts: list[int] = [len(run.entries) for run in runs]
complete_runs: int = sum(1 for run in runs if run.complete)
logger.info(
    "Runs: %d | cuts per run min/mean/max = %d/%.2f/%d | complete runs %d/%d",
    RUN_COUNT,
    min(cut_counts),
    sum(cut_counts) / len(cut_counts),
    max(cut_counts),
    complete_runs,
    RUN_COUNT,
)
logger.info("Distinct pairs cut across runs: %d", len(stats))
report_stats(stats, RUN_COUNT)
show_frequency_histogram(stats, RUN_COUNT)

consensus: list[PairStats] = consensus_pairs(stats, 0.9)
residual_cycles: int = verify_acyclic(cut_graph, [stat.canon for stat in consensus])
coverage: str = (
    "graph stays acyclic"
    if residual_cycles == 0
    else "leaves %d cyclic SCC(s)" % residual_cycles
)
logger.info(
    "Consensus at >= 90%% of runs: %d pair(s), %s", len(consensus), coverage
)

for label, pairs in (
    (">=90% of runs", consensus),
    (">=50% of runs", consensus_pairs(stats, 0.5)),
    (">=30% of runs", consensus_pairs(stats, 0.3)),
    ("all cut pairs", stats),
):
    logger.info("%s: %d pair(s), showing top %d", label, len(pairs), min(len(pairs), SHOW_TOP))
    show_pair_grid(cut_graph, pairs, SHOW_TOP, label)

logger.info("Random replay analysis done", start_timer=t0)